# Extended Toolkit for Data Extraction, Structuring, and Cross-Modal Alignment of Data from Bonsai Workflows

*A user-facing tutorial covering the full pipeline from raw Bonsai/Harp logs
and Neuropixel recordings to a unified results dictionary in xarray format.*

## **The Problem with `harp-python`**

Harp-python provides a low-level API for reading binary data from Harp devices:
```python
reader = harp.create_reader('device.yml')
df = reader.DigitalInputState.read('Behavior0/Behavior0_32_1904-01-01T02-00-00.bin')
```

This gives you **one DataFrame for one register from one device**. But in practice:

- **Multiple devices:** Experimental setups often have multiple HARP devices. Reading them all requires repeating the above for every register × every device.
- **Manual register lookup:** `harp-python` requires register *names* (e.g. `DigitalInputState`), not the *addresses* (e.g. `32`) that appear in folder names — users must cross-reference the YAML schema manually.
- **No multi-device coordination:** Peripherals like nosepokes span multiple boards (e.g. 18 nosepokes across 6 boards). There is no built-in way to combine columns from different devices into a single unified structure.
- **No timestamp collation:** Each register has its own timestamps. Merging them into a common time axis is left entirely to the user.
- **No alignment to external streams:** Neuropixels, video, and DLC data run on different clocks. `harp-python` provides no tools for cross-modal time alignment.

**The result:** users end up writing hundreds of lines of per-session boilerplate code to achieve routine tasks.

## **What This Toolkit Adds**

This toolkit exploits the one thing that is consistent across all experimental sessions: **the Bonsai output directory structure**. Device data always lives at `[device]/[register]/columns`. By defining a mapping from logical names (e.g. "Nosepoke0") to physical coordinates (device, register, localID), we can automate the entire pipeline.

The toolkit is organised into layers, each building on the last:

| Layer | Module | What It Does |
|---|---|---|
| **Data Extraction** | `HarpExtender` | Batch-read all devices/registers into organised nested dicts |
| **Timestamps** | `Timestamps` | Collect & merge timestamps from any level of that dict |
| **Data Structuring** | `LookupArrays` | Map logical names → physical coordinates in xarray; query with `ulookup` |
| **High-Level Classes** | `MultiDevices` / `Devices` | Declarative classes that do all the above and produce xarray DataArrays in one call |
| **Non-Binary Files** | `Filetypes` | Load session files (events CSV, video CSV, visual environment) in the same pattern |
| **Multi-Modal Alignment** | `Alignment_Utils` | Sync-pulse extraction, NPX↔Bonsai time conversion, video frame timestamps, global clock |

The end result: **all your Bonsai/Harp, Neuropixel, and video/DLC data in a common results dict, in xarray format, with aligned times.**



# **Part 1: Setup**
-----

In [2]:
#======== Imports ===============================================================
import os
import numpy as np
import pandas as pd
import xarray as xr
import harp
from pathlib import Path

# Toolkit imports — MultiDevices & Devices
from Refactor.MultiDevices.multidevice_mod import Nosepoke
from Refactor.Devices.devices import SoundCard, Camera0Frames

# Toolkit imports — Filetypes
from Refactor.Filetypes.filetypes import ExperimentEvents, VideoData, VisualEnvironment

# Toolkit imports — LookupArrays
from Refactor.LookupArrays.Lookup_Arrays_mod import (
    construct_data_array,
    construct_lookup_array,
    ulookup,
    update_data_array,
)

# Toolkit imports — Alignment Utils
from Refactor.Alignment_Utils.alignment_utils import (
    DigitalSyncPulse,
    Npx_SyncPulse,
    get_aligned_pulse_dfs,
    get_npx_to_bonsai_time_conversion,
    convert_npx_to_bonsai_time,
    plot_stacked_pulses,
    create_global_clock,
    index_map_util,
    get_video_timestamps,
)

print("All imports successful.")

All imports successful.


In [3]:
#======== User-Configurable Paths ==============================================
experiment_directory_path = './Bonsai_logs/2025-10-07T18-08-45'
harp_device_yaml_path     = './device.yml'
soundcard_yaml_path       = './soundcard.yml'
npx_sync_path             = './Bonsai_logs/JPVA191B_07102025_run2_g0/sync_channel.npy'

#======== Rig Layout: Devices ==================================================
# For this tutorial we use just two Behaviour boards for simplicity.
# Each board has three nosepokes connected to it → 6 nosepokes total.

device_list = ['Behavior0', 'Behavior1']

device_IDs = {
    'Behavior0': 'ID_0',
    'Behavior1': 'ID_1',
}

device_registers_dict = {
    'Behavior0': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
    'Behavior1': ['8', '32', '34', '35', '44', '49', '50', '51', '78', '92'],
}

#======== Rig Layout: Nosepoke Channel Definitions =============================
# Each nosepoke peripheral has a global name (e.g. "Nosepoke0") and maps to
# a specific (device, register, localID) tuple — the "virtual coordinates".
# This mapping mirrors the directory structure: dfs_dict[device][register][localID].

channel_list = [
    'Nosepoke0', 'Nosepoke1', 'Nosepoke2',   # → Behavior0
    'Nosepoke3', 'Nosepoke4', 'Nosepoke5',   # → Behavior1
]

# LocalIDs for each data type (column names inside the register DataFrames)
Nosepoke_Activations_localIDs     = ['DIPort0', 'DIPort1', 'DIPort2']
Nosepoke_LED_Activations_localIDs = ['DOPort0', 'DOPort1', 'DOPort2']
Nosepoke_Valve_Activations_localIDs = ['SupplyPort0', 'SupplyPort1', 'SupplyPort2']
Nosepoke_Reward_Release_localIDs  = ['PulseSupplyPort0', 'PulseSupplyPort1', 'PulseSupplyPort2']

channel_type_localIDs = {
    'Activations':     Nosepoke_Activations_localIDs,
    'LED_Activations': Nosepoke_LED_Activations_localIDs,
    'Valve_Activations': Nosepoke_Valve_Activations_localIDs,
    'Reward_Release':  Nosepoke_Reward_Release_localIDs,
}

# Which register address each localID lives in
Activations_localID_register_dict = {'DIPort0': '32', 'DIPort1': '32', 'DIPort2': '32'}
LED_Activations_localID_register_dict = {'DOPort0': '34', 'DOPort1': '34', 'DOPort2': '34'}
Valve_Activations_localID_register_dict = {'SupplyPort0': '34', 'SupplyPort1': '34', 'SupplyPort2': '34'}
Reward_Release_localID_register_dict = {'PulseSupplyPort0': '49', 'PulseSupplyPort1': '50', 'PulseSupplyPort2': '51'}

channel_type_registerIDs = {
    'Activations':     Activations_localID_register_dict,
    'LED_Activations': LED_Activations_localID_register_dict,
    'Valve_Activations': Valve_Activations_localID_register_dict,
    'Reward_Release':  Reward_Release_localID_register_dict,
}

print(f"Devices: {device_list}")
print(f"Channels: {channel_list}")
print(f"Data types: {list(channel_type_localIDs.keys())}")

Devices: ['Behavior0', 'Behavior1']
Channels: ['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5']
Data types: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']


# **Part 2: Nosepoke Data via `Nosepoke` (MultiDevice)**
-----

### The Global → Virtual Coordinate Mapping

The core idea of this toolkit is simple: **map logical names to physical locations in the directory tree**.

Every Bonsai session produces the same directory structure:
```
experiment_directory/
    Behavior0/
        Behavior0_32_*.bin    → DataFrame with columns [DIPort0, DIPort1, DIPort2, ...]
        Behavior0_34_*.bin    → DataFrame with columns [DOPort0, DOPort1, DOPort2, ...]
        ...
    Behavior1/
        Behavior1_32_*.bin    → same structure
        ...
```

The data for a single nosepoke activation lives at `dfs_dict[device][register][localID]` — e.g. `dfs_dict['Behavior0']['32']['DIPort0']` is Nosepoke0's activation data.

We define a **`virtual_map`** that encodes this relationship:
```python
{
    'Nosepoke0': {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort0'},
    'Nosepoke1': {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort1'},
    ...
}
```

The `Nosepoke` class takes one `virtual_map` per data type (Activations, LEDs, Valves, Rewards), reads all the raw data, collects timestamps, builds xarray DataArrays, and populates them — all automatically.

In [4]:
#======== Build virtual_maps ====================================================
# One virtual_map per data type. Each maps a global channel name to its
# physical (device, register, localID) coordinates.

def build_virtual_maps(channel_list, device_list, channel_type_localIDs, channel_type_registerIDs):
    """Build a dict of {data_key: virtual_map} from the rig layout dicts."""
    n_ports = len(list(channel_type_localIDs.values())[0])  # ports per device (3)
    virtual_maps = {}

    for type_key, localIDs in channel_type_localIDs.items():
        registerID_dict = channel_type_registerIDs[type_key]
        vmap = {}
        for ch_idx, channel_name in enumerate(channel_list):
            dev_idx  = ch_idx // n_ports          # which device
            port_idx = ch_idx % n_ports           # which port on that device
            device   = device_list[dev_idx]
            localID  = localIDs[port_idx]
            register = registerID_dict[localID]
            vmap[channel_name] = {'device': device, 'register': register, 'localID': localID}
        virtual_maps[type_key] = vmap

    return virtual_maps

virtual_maps = build_virtual_maps(channel_list, device_list, channel_type_localIDs, channel_type_registerIDs)

# Inspect one of the maps
print("Activations virtual_map:")
for ch, coords in virtual_maps['Activations'].items():
    print(f"  {ch} → {coords}")

Activations virtual_map:
  Nosepoke0 → {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort0'}
  Nosepoke1 → {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort1'}
  Nosepoke2 → {'device': 'Behavior0', 'register': '32', 'localID': 'DIPort2'}
  Nosepoke3 → {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort0'}
  Nosepoke4 → {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort1'}
  Nosepoke5 → {'device': 'Behavior1', 'register': '32', 'localID': 'DIPort2'}


In [5]:
#======== Instantiate Nosepoke ==================================================
# This single call internally:
#   1. Reads all binary register data via HarpExtender (collect_device_dfs)
#   2. Collects & merges timestamps across devices/registers
#   3. Builds an xr.DataArray per data_key with global coord + virtual coord dims
#   4. Builds a lookup array per data_key for querying
#   5. Populates DataArrays with actual values from the raw DataFrames

nosepoke = Nosepoke(
    virtual_maps          = virtual_maps,
    global_coord_name     = 'channel',
    virtual_coord_names   = ['device', 'register', 'localID'],
    dict_of               = 'dicts',
    data_keys             = list(channel_type_localIDs.keys()),
    experiment_directory_path = experiment_directory_path,
    harp_device_yaml_path    = harp_device_yaml_path,
    device_type           = 'Behavior',
    device_list           = device_list,
    device_IDs            = device_IDs,
    device_registers_dict = device_registers_dict,
    fill_value            = False,
    verbose               = False,
)

print(f"Data arrays built: {list(nosepoke.data_arrays.keys())}")
print(f"Lookup arrays built: {list(nosepoke.lookup_arrays.keys())}")

Data arrays built: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']
Lookup arrays built: ['Activations', 'LED_Activations', 'Valve_Activations', 'Reward_Release']


In [7]:
#======== Inspect Nosepoke Outputs ==============================================

# 1. The DataArrays — one per data type, dims = (Time, channel)
activations_da = nosepoke.data_arrays['Activations']
print(f"Activations DataArray: {activations_da.dims}, shape {activations_da.shape}")
display(activations_da)

# 2. The Lookup Array — maps channel names to their physical coords
activations_lookup = nosepoke.lookup_arrays['Activations']
print(f"\nLookup Array:")
display(activations_lookup.to_dataframe().reset_index())

# 3. Use ulookup to query: "which channels are on Behavior0?"
behavior0_channels = ulookup(
    activations_lookup,
    global_coord_name='channel',
    device='Behavior0'
)
print(f"\nChannels on Behavior0: {behavior0_channels}")

# 4. Use ulookup to query: "which channels use register 32?"
reg32_channels = ulookup(
    activations_lookup,
    global_coord_name='channel',
    register='32'
)
print(f"Channels on register 32: {reg32_channels}")

Activations DataArray: ('Time', 'channel'), shape (14, 6)


<xarray.DataArray 'Activations_data' (Time: 14, channel: 6)> Size: 672B
array([[0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])
Coordinates:
  * Time      (Time) float64 112B 1.056e+05 1.056e+05 ... 1.06e+05 1.06e+05
  * channel   (channel) <U9 216B 'Nosepoke0' 'Nosepoke1' ... 'Nosepoke5'
    device    (channel) <U9 216B 'Behavior0' 'Behavior0' ... 'Behavior1'
    register  (channel) <U2 48B '32' '32' '32' '32' '32' '32'
    localID   (channel) <U7 168B 'DIPort0' 'DIPort1' ... 'DIPort1' 'DIPort2'
Attributes:
    description:  Base DataArray for unified coordinates of type Activations_...
    source:       Constructed using construct_base_da function


Lookup Array:


,channel,device,register,localID,Activations_lookup
0,Nosepoke0,Behavior0,32,DIPort0,True
1,Nosepoke1,Behavior0,32,DIPort1,True
2,Nosepoke2,Behavior0,32,DIPort2,True
3,Nosepoke3,Behavior1,32,DIPort0,True
4,Nosepoke4,Behavior1,32,DIPort1,True
5,Nosepoke5,Behavior1,32,DIPort2,True



Channels on Behavior0: ['Nosepoke0', 'Nosepoke1', 'Nosepoke2']
Channels on register 32: ['Nosepoke0', 'Nosepoke1', 'Nosepoke2', 'Nosepoke3', 'Nosepoke4', 'Nosepoke5']


# **Part 3: Single-Device Data via `Device` Subclasses**
-----

The `Nosepoke` class (a `MultiDevice`) is for peripherals that span multiple boards and need the global→virtual mapping. For simpler devices that don't need that mapping — like SoundCards or individual camera registers — use the `Device` base class or its pre-built subclasses.

These classes take the same paths and device definitions, read all specified registers, and produce a dict of `xr.DataArray`s directly from the raw DataFrames. No virtual maps needed.

In [ ]:
#======== SoundCard =============================================================
# Pre-built subclass with sensible defaults for SoundCard registers (32, 33, 35).
# Only needs the experiment directory path.

soundcard = SoundCard(experiment_directory_path=experiment_directory_path)

print(f"SoundCard DataArrays: {list(soundcard.device_da_dict.keys())}")
for name, da in soundcard.device_da_dict.items():
    print(f"  {name}: shape {da.shape}")
display(soundcard.device_da_dict['PlaySoundFreq'])

SoundCard DataArrays: ['PlaySoundFreq', 'StopLog', 'AttenuationRight']


AttributeError: 'Dataset' object has no attribute 'shape'

In [ ]:
#======== Camera0Frames =========================================================
# Reads register 92 (Camera0Frame) from all Behaviour boards.
# Each board's camera produces a DataArray of frame timestamps.
# We'll use this later for video alignment.

camera_frames = Camera0Frames(experiment_directory_path=experiment_directory_path)

print(f"Camera0Frames DataArrays: {list(camera_frames.device_da_dict.keys())}")
for name, da in camera_frames.device_da_dict.items():
    n_frames = da.sizes.get('Time', len(da))
    print(f"  {name}: {n_frames} frames")

# **Part 4: Non-Binary Session Files via `Filetypes`**
-----

Alongside HARP binary data, Bonsai workflows produce CSV, JSONL, and YAML files for experiment events, video metadata, visual environments, and session settings.

The `FileTypeData` base class and its subclasses (`ExperimentEvents`, `VideoData`, `VisualEnvironment`, etc.) auto-locate and load these files from the experiment directory using the same device_type prefix pattern. Each produces a `.df` attribute containing the loaded DataFrame.

In [ ]:
#======== ExperimentEvents ======================================================
# Loads the ExperimentEvents CSV. Automatically renames 'Value' → 'Event'
# and sets the index to 'Time'.

events = ExperimentEvents(experiment_directory_path=experiment_directory_path)
print(f"ExperimentEvents: {events.df.shape[0]} events")
display(events.df.head(10))

In [ ]:
#======== VideoData =============================================================
# Loads the VideoData CSV with frame IDs and timestamps.

video_data = VideoData(experiment_directory_path=experiment_directory_path, file_type='csv')
print(f"VideoData: {video_data.df.shape[0]} rows")
display(video_data.df.head())

#======== VisualEnvironment ====================================================
# Loads the VisualEnvironment CSV with landmark/gratings/firefly columns.

vis_env = VisualEnvironment(experiment_directory_path=experiment_directory_path, file_type='csv')
print(f"\nVisualEnvironment: {vis_env.df.shape[0]} rows, columns: {list(vis_env.df.columns)}")
display(vis_env.df.head())

## Part 5 — LookupArrays: the indexing engine

Every `MultiDevice` internally builds two xarray objects per data key:

| Object | What it stores | Shape |
|--------|---------------|-------|
| **Data array** | The actual sensor values (e.g. activation booleans) | `(time, global_peripherals)` |
| **Lookup array** | A boolean matrix that maps each *global* peripheral → its *virtual* coordinates (device, register, localID) | `(global_peripherals, virtual_coord_levels)` |

The function `ulookup(lookup_array, **kwargs)` queries the lookup array to find
which global indices match a set of virtual coordinates — so you never need to
remember raw column numbers.

Below we show the standalone functions that `Nosepoke` calls under the hood.

In [ ]:
# ---- Retrieve the lookup array that Nosepoke already built ----
# nosepoke.lookup_arrays is a dict of {data_key: (lookup_da, lookup_virtual_coords)}

activations_lookup, activations_vcoords = nosepoke.lookup_arrays['Activations']

print("Lookup array shape:", activations_lookup.shape)
print("Dims:", activations_lookup.dims)
print("Coords:")
for c in activations_lookup.coords:
    print(f"  {c}: {activations_lookup.coords[c].values}")
print()

# ---- ulookup: query by virtual coordinates ----
# "Which global peripheral indices correspond to Behavior0, register 32?"
matched_indices = ulookup(activations_lookup, device='Behavior0', register='32')
print(f"ulookup(device='Behavior0', register='32') → global indices: {matched_indices}")

# "All peripherals on Behavior1?"
matched_b1 = ulookup(activations_lookup, device='Behavior1')
print(f"ulookup(device='Behavior1')                → global indices: {matched_b1}")

In [ ]:
# ---- Building blocks: construct_data_array & construct_lookup_array ----
# These are the low-level functions that MultiDevice calls internally.
# You would only use these directly for a custom device not covered by a
# subclass.

# construct_data_array: merges per-device DataFrames into one xarray along
# a global coordinate axis.
# construct_lookup_array: builds the boolean mapping from global → virtual.

# Example: manually build a lookup array from the Activations virtual_map.
from Refactor.LookupArrays.Lookup_Arrays_mod import construct_lookup_array

activations_vmap = nosepoke.virtual_maps['Activations']
print("Virtual map for 'Activations':")
for k, v in activations_vmap.items():
    print(f"  global {k}: {v}")

manual_lookup, manual_vcoords = construct_lookup_array(activations_vmap)
print(f"\nManually built lookup array shape: {manual_lookup.shape}")
print(f"Virtual coord names: {list(manual_vcoords.keys())}")

## Part 6 — Alignment Utils: synchronising Bonsai and Neuropixel clocks

### Why alignment is needed

Bonsai/Harp and Neuropixel run on **independent clocks** that drift over time.
To combine behavioural events (Harp timestamps) with neural data (NPX sample
indices) we need a shared time reference.

**How it works:**

1. A **TTL square-wave pulse train** is generated and recorded on both systems
   simultaneously — one copy on a Harp Behavior board (digital IO), one on an
   NPX sync channel.
2. `DigitalSyncPulse` extracts the Bonsai-side pulse start/end times (in
   seconds) from DO1 rise/fall registers (34 & 35) of the sync device.
3. `Npx_SyncPulse` extracts the NPX-side pulse durations from the raw `.npy`
   recording (in sample units, converted via `npx_frequency`).
4. `get_aligned_pulse_dfs` trims both pulse trains to the same count and
   normalises t = 0 to the first pulse.
5. `get_npx_to_bonsai_time_conversion` computes a single linear conversion
   ratio: $r = \frac{\Delta t_{\text{NPX}}}{\Delta t_{\text{Bonsai}}}$. It
   also produces error diagnostics (scatter + histogram) so you can verify that
   drift is within tolerance.
6. `convert_npx_to_bonsai_time` applies this ratio to any NPX DataFrame,
   Series, or array to express it in Bonsai seconds.

In [ ]:
# ---- Step 1: Extract Bonsai-side sync pulses ----
# The TTL pulse train was recorded on Behavior3's DO1 rise (reg 34) and
# fall (reg 35) registers.  align_to_zero=True so the first pulse starts
# at t = 0 — we only care about *relative* timing for the conversion.

sync_pulses = DigitalSyncPulse(
    experiment_directory_path = experiment_directory_path,
    harp_device_yaml_path     = harp_device_yaml_path,
    Sync_device               = 'Behavior3',
    align_to_zero             = True,
    verbose                   = False,
)

print(f"Bonsai sync pulses: {len(sync_pulses.SyncPulseTimes)} detected")
print(f"Initial start offset (original timebase): {sync_pulses.initial_start_offset:.4f} s")
display(sync_pulses.SyncPulseTimes.head())

In [ ]:
# ---- Step 2: Extract NPX-side sync pulses ----
# The same TTL pulse train was recorded on the Neuropixel sync channel.
# npx_frequency converts raw sample indices → seconds.
# align_to_zero=False because we want the raw times for the conversion.

npx_sync = Npx_SyncPulse(
    npx_path       = npx_sync_path,
    npx_frequency  = 30000,          # 30 kHz NPX sampling rate
    align_to_zero  = False,
    verbose        = False,
)

print(f"NPX sync pulses: {len(npx_sync.npx_pulse_durations)} detected")
display(npx_sync.npx_pulse_durations.head())

### Step 3 — Align & convert

`get_npx_to_bonsai_time_conversion` internally calls `get_aligned_pulse_dfs`
to trim both pulse trains to the same count and normalise t = 0, then computes
the conversion ratio $r$.  Passing `visualisation=True` shows the overlaid
pulse trains; `error_visualisation_tools=True` adds per-pulse error scatter
plots and histograms so you can verify sub-millisecond alignment.

In [ ]:
# ---- Step 3: Compute conversion ratio with diagnostics ----

aligned_npx_converted, npx_conversion_ratio = get_npx_to_bonsai_time_conversion(
    sync_df                  = sync_pulses.SyncPulseTimes,
    npx_df                   = npx_sync.npx_pulse_durations,
    visualisation            = True,      # overlaid pulse trains
    error_visualisation_tools = True,     # per-pulse error scatter + histograms
    verbose                  = True,
)

print(f"\nConversion ratio (NPX_time / Bonsai_time): {npx_conversion_ratio:.10f}")
print(f"To go from NPX → Bonsai:  bonsai_t = (npx_t - npx_start) / ratio + bonsai_start")

In [ ]:
# ---- Step 4: Convert arbitrary NPX data → Bonsai time ----
# convert_npx_to_bonsai_time is a convenience wrapper that accepts
# DataFrames, Series, or plain arrays.  Pass the two pulse objects and
# it computes the ratio automatically.

npx_pulse_df_in_bonsai_time = convert_npx_to_bonsai_time(
    npx_data       = npx_sync.npx_pulse_durations,
    npx_sync_pulse = npx_sync,
    sync_pulses    = sync_pulses,
    verbose        = False,
)

print("NPX pulse durations converted to Bonsai seconds:")
display(npx_pulse_df_in_bonsai_time.head())

## Part 7 — Video timestamps & the global clock

### Video timestamps

`get_video_timestamps` uses `sleap_io` to read the video frame count, then
auto-matches it against the Camera0Frames harp data to assign a Bonsai-time
timestamp to every frame.  This requires `sleap_io` to be installed and a
video file on disk.

> If `sleap_io` is unavailable, you can fall back to `Camera0Frames` directly
> (shown in Part 3) — the frame-count matching just won't be automatic.

### Global clock

`create_global_clock` builds a uniformly-spaced time axis (e.g. at 1 ms
resolution) that serves as the shared index for all data streams.
`index_map_util` then maps each stream's irregular timestamps onto the
nearest global-clock bin, producing a `(N × 5)` matrix of
`[global_idx, stream_idx, global_time, stream_time, Δt]`.

In [ ]:
# ---- Video timestamps (requires sleap_io) ----
# Uncomment the block below if sleap_io is installed and you have a video file.

video_file_path = os.path.join(experiment_directory_path,
                               'VideoData', 'VideoData_1904-01-02T05-00-00.avi')

try:
    frame_times_da = get_video_timestamps(
        experiment_directory_path = experiment_directory_path,
        video_path               = video_file_path,
        auto_match_device        = True,
        harp_device_yaml_path    = harp_device_yaml_path,
        verbose                  = True,
    )
    print(f"\nVideo frame timestamps: {frame_times_da.shape} DataArray")
    display(frame_times_da[:5])

except ImportError:
    print("sleap_io not installed — falling back to Camera0Frames from Part 3.")
    print("Use camera0frames.device_da_dict to get frame timestamps per device.")
    frame_times_da = None

except Exception as e:
    print(f"Could not load video timestamps: {e}")
    frame_times_da = None

In [ ]:
# ---- Global clock & index mapping ----
# Build a 1 ms resolution global clock spanning the sync pulse window.
# Then map one of our harp streams onto it.

bonsai_start = sync_pulses.initial_start_offset   # original session start
bonsai_end   = bonsai_start + sync_pulses.SyncPulseTimes['End'].iloc[-1]

global_clock = create_global_clock(
    start_time         = 0.0,
    end_time           = bonsai_end - bonsai_start,  # relative time
    timestep_interval  = 0.001,                      # 1 ms bins
    include_end_time   = True,
    verbose            = True,
)

# Map the Activations xarray time axis onto the global clock.
activations_da = nosepoke.data_arrays['Activations']
stream_times   = activations_da.coords['Time'].values - bonsai_start  # relative

times_matrix, index_matrix, full_matrix = index_map_util(
    global_clock_times = global_clock,
    stream_times       = stream_times,
    match_type         = 'Nearest',
)

print(f"\nGlobal clock: {len(global_clock)} bins ({global_clock[0]:.3f} → {global_clock[-1]:.3f} s)")
print(f"Activations stream: {len(stream_times)} samples")
print(f"full_matrix shape: {full_matrix.shape}  (cols: global_idx, stream_idx, global_t, stream_t, Δt)")
print(f"Max |Δt| = {np.nanmax(np.abs(full_matrix[:, 4])):.6f} s")

## Part 8 — Unified results dictionary

The goal of the whole pipeline is a single `results` dictionary where every
data stream — Bonsai/Harp, Neuropixel, Filetypes, and video — lives in a
common xarray-compatible format with aligned time references.

Below we assemble `results` from the objects created in Parts 2–7.

In [ ]:
# ---- Assemble the unified results dict ----

results = {}

# --- Bonsai / Harp: MultiDevice data arrays + lookup arrays ---
for dk in nosepoke.data_arrays:
    results[f'nosepoke_{dk}'] = {
        'data':   nosepoke.data_arrays[dk],            # xr.DataArray (Time × peripherals)
        'lookup': nosepoke.lookup_arrays[dk][0],       # xr.DataArray (peripherals × virtual coords)
    }

# --- Bonsai / Harp: Single-device data arrays ---
for dk, da in soundcard.device_da_dict.items():
    results[f'soundcard_{dk}'] = {'data': da}

for dk, da in camera0frames.device_da_dict.items():
    results[f'camera_{dk}'] = {'data': da}

# --- Filetypes ---
results['experiment_events'] = {'data': experiment_events.df}
results['video_data']        = {'data': video_data.df}
results['visual_environment'] = {'data': vis_env.df}

# --- Neuropixel alignment ---
results['sync_alignment'] = {
    'bonsai_sync_pulses': sync_pulses.SyncPulseTimes,
    'npx_sync_pulses':    npx_sync.npx_pulse_durations,
    'npx_converted':      aligned_npx_converted,
    'conversion_ratio':   npx_conversion_ratio,
    'bonsai_start_offset': sync_pulses.initial_start_offset,
}

# --- Global clock & index map ---
results['global_clock'] = {
    'times':       global_clock,
    'index_map':   full_matrix,           # (N × 5) for Activations stream
    'timestep_ms': 1.0,
}

# --- Video timestamps (if available) ---
if frame_times_da is not None:
    results['video_timestamps'] = {'data': frame_times_da}

print(f"Results dict assembled — {len(results)} entries:")
for k, v in results.items():
    if isinstance(v, dict) and 'data' in v:
        d = v['data']
        shape = getattr(d, 'shape', getattr(d, 'dims', len(d)))
        print(f"  {k:30s}  shape={shape}")
    else:
        print(f"  {k:30s}  keys={list(v.keys())}")

---

## Summary

This notebook walked through the full pipeline from raw Bonsai/Harp logs and
Neuropixel recordings to a unified `results` dictionary:

| Step | Module | What it does |
|------|--------|-------------|
| **Setup** | — | Define paths, device list, nosepoke channel maps |
| **Part 2** | `MultiDevices` → `Nosepoke` | Loads multi-board harp data, merges into xarray, builds virtual→global lookup |
| **Part 3** | `Devices` → `SoundCard`, `Camera0Frames` | Single-device wrappers with sensible register defaults |
| **Part 4** | `Filetypes` → `ExperimentEvents`, `VideoData`, `VisualEnvironment` | CSV/binary file loaders with column renaming |
| **Part 5** | `LookupArrays` → `ulookup`, `construct_lookup_array` | Boolean indexing engine: query by virtual coords |
| **Part 6** | `Alignment_Utils` → `DigitalSyncPulse`, `Npx_SyncPulse`, `get_npx_to_bonsai_time_conversion` | TTL sync pulse extraction, alignment, conversion ratio with error diagnostics |
| **Part 7** | `Alignment_Utils` → `get_video_timestamps`, `create_global_clock`, `index_map_util` | Frame-level video timestamps, uniform global time axis, nearest-bin mapping |
| **Part 8** | — | Assemble everything into one `results` dict |

**Next steps:** Use the `results` dict to index into any stream by its global
clock bin, overlay neural and behavioural events on the same time axis, or
export to downstream analysis tools (e.g. NWB, DeepLabCut).